# AllMusicCaps CLAP (MTG) vs LAION / Microsoft CLAP

Try MTG's music-trained CLAP on the test set (Traktor rating 4-5 with a colour) and compare it with the two models
from [text_search.ipynb](text_search.ipynb). Background and caveats: [allmusiccaps_clap.md](../../allmusiccaps_clap.md).

| Key | Model |
|---|---|
| `laion-unfused` | LAION `clap-htsat-unfused` (the app's model) |
| `ms-2023` | Microsoft CLAP 2023 |
| `amc-all-layers` | `mtg-upf/allmusiccaps_all_layers`: OMAR-RQ, frozen text encoder |
| `amc-te-sigreg` | `mtg-upf/allmusiccaps_te_trained_sigreg`: trained text encoder, MTG's best |

**Setup** (once; the venv is `.research_cache\venv-clap`, kernel *SongSelector CLAP (venv-clap)*):

```
.research_cache\venv-clap\Scripts\python -m pip install amclap==0.1.0
.research_cache\venv-clap\Scripts\python docs\research\experiments\clap\extract_embeddings.py --app-data dist --models amc-all-layers amc-te-sigreg
```

`amclap` pins `sentence-transformers<5.2`; the venv gets 5.1.2, the system Python keeps its own. The extraction embeds
every 10 s window at 24 kHz (about 20 ms per window on the GPU) and is resumable: re-run the first cell to pick up
tracks finished since.

**Licence:** weights CC BY-NC-SA 4.0, code AGPL-3.0, non-commercial research.

How the numbers are made: every window vector is L2-normalised, a track is the mean of its windows (normalised again),
the score is the cosine between that and the prompt. Scores are not comparable *between* models, only rankings are.

In [ ]:
import os
import re
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'pyproject.toml').exists():
    REPO = REPO.parent
os.environ['HF_HOME'] = str(REPO / '.research_cache' / 'hf')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
sys.path.insert(0, str(REPO / 'docs' / 'research' / 'experiments' / 'clap'))

from benchmark_cpu import BATCH, decode, load_model, windows_of  # noqa: E402 (needs the sys.path entry above)

CLAP = REPO / '.research_cache' / 'clap'
MODEL_KEYS = ['laion-unfused', 'ms-2023', 'amc-all-layers', 'amc-te-sigreg']
with sqlite3.connect(CLAP / 'library.sqlite') as connection:
    library = pd.read_sql('SELECT path_key, file_path, artist, title, genre, bpm FROM tracks', connection)
library = library.set_index('path_key')


def unit(matrix):
    return matrix / np.linalg.norm(matrix, axis=-1, keepdims=True)


def window_vectors(model, path_key):
    """(n, dim) normalised window vectors of one track, and the window length in seconds."""
    with sqlite3.connect(CLAP / f'{model}.sqlite') as connection:
        n, dim, window_s, blob = connection.execute(
            'SELECT n, dim, window_s, vectors FROM windows WHERE path_key = ?', (path_key,)).fetchone()
    return unit(np.frombuffer(blob, dtype=np.float32).reshape(n, dim)), window_s


def track_vectors(model):
    """{path_key: track vector}: the mean of the normalised window vectors, normalised."""
    with sqlite3.connect(CLAP / f'{model}.sqlite') as connection:
        rows = connection.execute('SELECT path_key, n, dim, vectors FROM windows').fetchall()
    return {key: unit(unit(np.frombuffer(blob, dtype=np.float32).reshape(n, dim)).mean(0))
            for key, n, dim, blob in rows}


available = [m for m in MODEL_KEYS if (CLAP / f'{m}.sqlite').exists()]
per_model = {m: track_vectors(m) for m in available}
# Compare on the tracks every model has, so no model wins by having a different pool.
common = sorted(set.intersection(*(set(v) for v in per_model.values())))
tracks = {m: np.stack([per_model[m][key] for key in common]) for m in available}
pd.Series({**{m: len(v) for m, v in per_model.items()}, 'common': len(common)}, name='tracks')

## Models on demand

Loaded the first time they are needed (about 20-40 s each). The audio half goes to the GPU when there is one.

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
loaded = {}


def model(key):
    if key not in loaded:
        loaded[key] = load_model(key)
        loaded[key].audio_to(DEVICE)
    return loaded[key]


def text_vector(key, prompt):
    return unit(model(key).embed_text([prompt])[0].astype(np.float32))


def label(path_key, width=48):
    row = library.loc[path_key]
    return ' - '.join(part for part in (row.artist, row.title) if part)[:width]

## Search

`search` ranks the common tracks for one model; `compare` puts the top `k` of every model side by side.

In [ ]:
def search(prompt, key='amc-te-sigreg', k=10):
    scores = tracks[key] @ text_vector(key, prompt)
    top = np.argsort(-scores)[:k]
    result = library.loc[[common[i] for i in top], ['artist', 'title', 'genre', 'bpm']].reset_index(drop=True)
    result.insert(0, 'score', scores[top].round(3))
    result.index += 1
    return result


def compare(prompt, keys=None, k=10):
    columns = {}
    for key in keys or available:
        scores = tracks[key] @ text_vector(key, prompt)
        top = np.argsort(-scores)[:k]
        columns[key] = [f'{label(common[i], 40)}  [{library.loc[common[i], "genre"] or "-"}]' for i in top]
    table = pd.DataFrame(columns)
    table.index += 1
    return table.style.set_properties(**{'text-align': 'left'}).set_caption(prompt)

In [ ]:
compare('dark rolling drum and bass with a heavy reese bassline')

In [ ]:
compare('A funky breakbeat track with chopped drum breaks and a groovy bassline.')

In [ ]:
search('calm downtempo track with warm pads and soft piano', 'amc-te-sigreg')

## A rough number: do the top results carry the right genre tag?

The labels are the DJ's own genre tags (`genre` column, comma-separated, a leading `-` means *not*). Pool: common tracks
with at least one tag. Three prompt styles per label: the bare tag, LAION's template, a review-style caption (the kind
AllMusicCaps was trained on).

- **P@10**: share of the top 10 that carry the tag. **AP**: average precision over the whole ranking (steadier).
- **chance**: share of the pool that carries the tag; AP of a random ranking is about this.
- Tags are sparse and inconsistent (an untagged downtempo track counts as a miss), so absolute numbers understate
  every model. Read the *differences between models*, and expect noise with small `n`.

In [ ]:
def positive_tags(genre):
    tokens = (t.strip().lower() for t in re.split(r'[,\\/;]', genre or ''))
    return [t for t in tokens if t and not t.startswith('-')]


LABELS = {  # label: (does a tag match?, prompts)
    'drum and bass': (lambda t: 'dnb' in t or 'drum' in t or 'jungle' in t,
                      ['drum and bass', 'This audio is a drum and bass song.',
                       'Fast, rolling drum and bass with deep sub-bass and sharp breakbeats.']),
    'breakbeat': (lambda t: 'break' in t,
                  ['breakbeat', 'This audio is a breakbeat song.',
                   'A funky breakbeat track with chopped drum breaks and a groovy bassline.']),
    'downtempo': (lambda t: 'down' in t,
                  ['downtempo', 'This audio is a downtempo song.',
                   'A laid-back downtempo track with slow beats and a warm, mellow atmosphere.']),
    'liquid': (lambda t: 'liquid' in t,
               ['liquid drum and bass', 'This audio is a liquid drum and bass song.',
                'Smooth, soulful liquid drum and bass with lush pads and melodic vocals.']),
    'neurofunk': (lambda t: 'neuro' in t,
                  ['neurofunk', 'This audio is a neurofunk song.',
                   'Dark, technical neurofunk with heavy distorted reese basslines.']),
    'jungle': (lambda t: 'jungle' in t,
               ['jungle', 'This audio is a jungle song.',
                'Old-school jungle with chopped amen breaks and booming reggae sub-bass.']),
    'psy': (lambda t: 'psy' in t,
            ['psychedelic breakbeat', 'This audio is a psychedelic breakbeat song.',
             'Trippy psychedelic breaks with acid squelches and hypnotic synth arpeggios.']),
    'electro': (lambda t: 'electro' in t and 'electronic' not in t,
                ['electro', 'This audio is an electro song.',
                 'Robotic electro with 808 drum machines and funky synth bass.']),
    'house': (lambda t: 'house' in t,
              ['house music', 'This audio is a house song.',
               'A four-on-the-floor house track with a steady kick and a groovy bassline.']),
}
STYLES = ['tag', 'template', 'caption']

tags = {key: positive_tags(library.loc[key, 'genre']) for key in common}
pool = np.array([bool(tags[key]) for key in common])
truth = {name: np.array([any(match(t) for t in tags[key]) for key in common])[pool]
         for name, (match, _) in LABELS.items()}


def average_precision(relevant_in_rank_order):
    hits = np.cumsum(relevant_in_rank_order)
    return float((hits / np.arange(1, len(hits) + 1))[relevant_in_rank_order].mean())


rows = []
for name, (_, prompts) in LABELS.items():
    relevant = truth[name]
    if relevant.sum() < 5:
        continue
    for style, prompt in zip(STYLES, prompts):
        row = {'label': name, 'style': style, 'n': int(relevant.sum()), 'chance': relevant.mean()}
        for key in available:
            order = np.argsort(-(tracks[key][pool] @ text_vector(key, prompt)))
            row[f'{key} P@10'] = relevant[order[:10]].mean()
            row[f'{key} AP'] = average_precision(relevant[order])
        rows.append(row)
genre_check = pd.DataFrame(rows).set_index(['label', 'style'])
print(f'pool: {pool.sum()} tagged tracks of {len(common)}')
genre_check.round(2)

In [ ]:
ap = genre_check[[f'{key} AP' for key in available]]
ap.columns = available
summary = pd.concat({'mean AP by label': ap.groupby(level='label').mean(),
                     'mean AP by prompt style': ap.groupby(level='style').mean()})
summary.loc[('all', 'mean'), :] = ap.mean()
summary.round(3).style.highlight_max(axis=1, props='font-weight: bold')

## Where in the track does a prompt match?

Per-window scores for one result, and the best-matching 10 s to listen to.

In [ ]:
import matplotlib.pyplot as plt
import soundfile
from IPython.display import Audio, display


def where(prompt, key='amc-te-sigreg', rank=1):
    scores = tracks[key] @ text_vector(key, prompt)
    path_key = common[np.argsort(-scores)[rank - 1]]
    windows, window_s = window_vectors(key, path_key)
    per_window = windows @ text_vector(key, prompt)
    best = int(np.argmax(per_window))
    plt.figure(figsize=(9, 2.2))
    plt.plot(np.arange(len(per_window)) * window_s / 60, per_window)
    plt.axvline(best * window_s / 60, color='grey', lw=0.8)
    plt.xlabel('minutes')
    plt.ylabel('cosine')
    plt.title(f'#{rank} {label(path_key, 60)} ({key})', fontsize=10)
    plt.show()
    path = library.loc[path_key, 'file_path']
    rate = soundfile.info(path).samplerate
    clip, rate = soundfile.read(path, start=int(best * window_s * rate), frames=int(window_s * rate), dtype='float32')
    display(Audio(clip.T, rate=rate))


where('dark rolling drum and bass with a heavy reese bassline')

## Zero-shot tags for any audio file

Embeds every window of a file (it need not be in the library) and ranks a tag vocabulary. Ranks only: the cosine
scale differs per model.

In [ ]:
VOCABULARY = ['drum and bass', 'liquid drum and bass', 'neurofunk', 'jungle', 'breakbeat', 'big beat', 'electro',
              'downtempo', 'trip hop', 'house', 'techno', 'trance', 'acid', 'dubstep', 'hip hop', 'ambient',
              'female vocals', 'male vocals', 'instrumental', 'piano', 'dark', 'uplifting', 'aggressive', 'chilled']


def embed_file(key, path):
    m = model(key)
    windows = windows_of(decode(path, m.rate), m.window)
    vectors = np.concatenate([m.embed_prepared(m.prepare(windows[i:i + BATCH])) for i in range(0, len(windows), BATCH)])
    return unit(unit(vectors.astype(np.float32)).mean(0))


def describe(path, keys=None, top=8):
    columns = {}
    for key in keys or available:
        vocabulary = unit(np.stack([text_vector(key, tag) for tag in VOCABULARY]))
        scores = vocabulary @ embed_file(key, path)
        columns[key] = [VOCABULARY[i] for i in np.argsort(-scores)[:top]]
    table = pd.DataFrame(columns)
    table.index += 1
    return table


describe(library.loc[common[0], 'file_path'])